Rapport TP comptage des mots
Johan Balduzzi INFO FISE 5A

Partie 1 : Python simple
L'objectif de cette partie est de compter le nombre d'occurence des mots dans le fichier mots.txt en utilisant un dictionnaire python.

On commence par générer le fichier mots.txt avec une liste de mots prédéfinis.

In [4]:
import random

comptes = {"chat":0,"chien":0,"arbre":0,"fleur":0,"maison":0,"soleil":0,
           "montagne":0,"livre":0,"ciel":0,"mer":0,"lune":0,"route":0,
           "oiseau":0,"jardin":0,"voiture":0}

for k in comptes.keys():
    comptes[k] = random.randint(1, 10)

mots = [m for m, n in comptes.items() for _ in range(n)]
random.seed(42)
random.shuffle(mots)

with open("mots.txt", "w", encoding="utf-8") as f:
    for i in range(0, len(mots)):
        f.write(mots[i] + "\n")

print("fichier crée")

fichier crée


On va maintenant compter le nombre d'occurence de chaque mots

In [9]:
%%time
def compter_occurences(fichier):
    with open(fichier, 'r', encoding='utf-8') as f:
        mots = f.read().split()

    total_mot = 0
    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
        total_mot += 1
    print("total = ", total_mot)
    return compteur

if __name__ == "__main__":
    fichier = "mots.txt"
    resultats = compter_occurences(fichier)
    for mot, nb in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {nb}")
    

total =  136
soleil : 20
jardin : 20
oiseau : 16
livre : 14
route : 10
arbre : 10
chat : 10
fleur : 8
chien : 8
lune : 6
montagne : 4
voiture : 4
maison : 2
mer : 2
ciel : 2
CPU times: user 466 μs, sys: 4.5 ms, total: 4.97 ms
Wall time: 3.24 ms


Partie 2 Pseudo-MapReduce 
L'objectif de cette partie est de reproduire les étapes du modèle MapReduce à l'aide de fonctions mapper, shuffle et reducer.

In [10]:
%%time

from collections import defaultdict

def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

if __name__ == "__main__":
    fichier = "mots.txt"
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f: 
            mapped.extend(mapper(ligne))
    grouped = shuffle(mapped)
    reduced = reducer(grouped)

    for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {count}")    
    

soleil : 20
jardin : 20
oiseau : 16
livre : 14
route : 10
arbre : 10
chat : 10
fleur : 8
chien : 8
lune : 6
montagne : 4
voiture : 4
maison : 2
mer : 2
ciel : 2
CPU times: user 0 ns, sys: 3.86 ms, total: 3.86 ms
Wall time: 3.11 ms


Partie 3 : Hadoop
L'objectif de cette partie est d'exécuter le WordCount sur HDFS en utilisant Hadoop

In [11]:
!hdfs dfs -mkdir -p /user/$USER/wordcount/input
!hdfs dfs -put -f mots.txt /user/$USER/wordcount/input/

!hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar wordcount /user/$USER/wordcount/input /user/$USER/wordcount/output

!hdfs dfs -cat /user/$USER/wordcount/output/part-r-00000

2026-10-13 13:28:09,803 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
org.apache.hadoop.mapred.FileAlreadyExistsException: Output directory hdfs://localhost:8020/user/johan/wordcount/output already exists
	at org.apache.hadoop.mapreduce.lib.output.FileOutputFormat.checkOutputSpecs(FileOutputFormat.java:164)
	at org.apache.hadoop.mapreduce.JobSubmitter.checkSpecs(JobSubmitter.java:277)
	at org.apache.hadoop.mapreduce.JobSubmitter.submitJobInternal(JobSubmitter.java:143)
	at org.apache.hadoop.mapreduce.Job$11.run(Job.java:1678)
	at org.apache.hadoop.mapreduce.Job$11.run(Job.java:1675)
	at java.base/java.security.AccessController.doPrivileged(Native Method)
	at java.base/javax.security.auth.Subject.doAs(Subject.java:423)
	at org.apache.hadoop.security.UserGroupInformation.doAs(UserGroupInformation.java:1899)
	at org.apache.hadoop.mapreduce.Job.submit(Job.java:1675)
	at org.apache.hadoop.mapreduce.Job.waitForCompletion(Job.java:1696)
	at org

Premiers Résultats : 

Le comptage de mots en python simple à pris 2.56ms ce qui est très rapide pour un comptage manuel car le fichier est petit.

La version pseudo-MapReduce à pris 0.90ms. cet écart s'explique par l'utilisation de la fonction print dans la version python simple pour compter le nombre total de mot. Déplacer cette étape peut réduire le temps d'exécution de la première version.

La méthode avec Hadoop à pris 47.2s ce qui est assez élevé. Le résultat est dû à une première utilisation de hadoop sous mon WSL2 pour faire tous les démarrages d'une première utilisation. Un nouvel essai à été réalisé et le temps d'exécution est de 9.9s ce qui est plus cohérent avec l'exercice. 

Nous allons maintenant augmenter la taille du fichier mots.txt. Pour cela, nous allons doubler le nombre d'occurence de chaques mots présents dans ce fichier.

In [7]:
with open("mots.txt", "r", encoding="utf-8") as f:
    contenu = f.read()

with open("mots.txt", "w", encoding="utf-8") as f:
    f.write(contenu)
    if not contenu.endswith("\n"):
        f.write("\n")
    f.write(contenu)

On renvoi le fichier sur HDFS pour que Hadoop traite la bonne version.

In [8]:
!hdfs dfs -put -f mots.txt /user/$USER/wordcount/input/

Second résultats : 

La version de python simple a un temps d'exécution de   4.97ms ce qui est presque le double du premier résultats obtenu. C'est cohérent en considérent qu'on à doublé la taille du fichier.

La version pseudo-MapReduce a un temps d'exécution de  3.86ms ce qui est presque le quadruple du premier résultat. La complexité alogorithmique de cette méthode peut expliquer cette augmentation du temps d'exécution de cette méthode.

La méthode Hadoop a un temps d'exécution de 9.3s ce qui ne représente pas une différence notable avec le premier résultat. On peut observer que le temps d'exécution est même plus court mais des expérience supplémentaires permettrai de déterminer si c'est causé par la taille du fichier ou un calibrage du système sur ce nouvel outil. 